In [ ]:
import polars as pl
import numpy as np
from statsmodels.regression.linear_model import OLS
def clean_cricket_data():
    """Load and clean cricket data properly"""
    
    # Load data
    cricket_df = pl.read_csv('data_cricket_runs.csv')
    
    # Filter for SL home, IND away, India batting
    filtered_df = cricket_df.filter(
        (pl.col('Home') == 'SL') & 
        (pl.col('Away') == 'IND') & 
        (pl.col('MatchInn') == 2)
    )
    
    print(f"Filtered matches: {len(filtered_df)}")
    
    # Get score columns (should be 300 columns for 50 overs × 6 balls)
    all_cols = cricket_df.columns
    score_cols = [col for col in all_cols 
                  if col not in ['MatchNbr', 'MatchInn', 'StartDate', 'Home', 'Away', 'team1', 'team2']]
    
    print(f"Score columns: {len(score_cols)}")
    
    # Extract target match
    target_df = filtered_df.filter(pl.col('MatchNbr') == 111404)
    historical_df = filtered_df.filter(pl.col('MatchNbr') != 111404)
    
    if len(target_df) == 0:
        print("Error: Target match not found!")
        return None
    
    print(f"Target match: {len(target_df)} record")
    print(f"Historical matches: {len(historical_df)}")
    
    # Extract and clean score data
    target_scores = target_df.select(score_cols).to_numpy()[0]
    historical_scores = historical_df.select(score_cols).to_numpy()
    
    # Clean data - convert to float and handle any non-numeric values
    def clean_scores(scores):
        cleaned = np.zeros_like(scores, dtype=float)
        for i, val in enumerate(scores):
            try:
                cleaned[i] = float(val) if val is not None else 0.0
            except (ValueError, TypeError):
                cleaned[i] = 0.0
        return cleaned
    
    target_trajectory = clean_scores(target_scores)
    
    historical_trajectories = np.zeros((len(historical_scores), len(score_cols)), dtype=float)
    for i, row in enumerate(historical_scores):
        historical_trajectories[i] = clean_scores(row)
    
    print(f"Cleaned target shape: {target_trajectory.shape}")
    print(f"Cleaned historical shape: {historical_trajectories.shape}")
    print(f"Target score at T0=150: {target_trajectory[149]:.1f}")
    print(f"Target final score: {target_trajectory[299]:.1f}")
    
    return target_trajectory, historical_trajectories
def synthetic_control_prediction(target_trajectory, historical_trajectories, T0=150, r_tilde=5):
    """
    Perform synthetic control with proper matrix handling
    """
    
    print(f"\nSYNTHETIC CONTROL PREDICTION")
    print("="*40)
    
    # Split into training and prediction periods
    target_train = target_trajectory[:T0]
    target_remaining = target_trajectory[T0:]
    
    historical_train = historical_trajectories[:, :T0]  # N × T0
    historical_remaining = historical_trajectories[:, T0:]  # N × (300-T0)
    
    print(f"Training period: {T0} time points")
    print(f"Prediction period: {len(target_remaining)} time points")
    print(f"Historical train shape: {historical_train.shape}")
    print(f"Historical remaining shape: {historical_remaining.shape}")
    
    # Transpose for OLS: X should be T0 × N, y should be T0 × 1
    X_train = historical_train.T  # T0 × N
    y_train = target_train       # T0
    
    print(f"OLS setup: X={X_train.shape}, y={y_train.shape}")
    
    # SVD filtering on X_train
    U, s, Vt = np.linalg.svd(X_train, full_matrices=False)
    
    print(f"SVD: U={U.shape}, s={s.shape}, Vt={Vt.shape}")
    print(f"Singular values: {s[:min(10, len(s))]}")
    
    # Threshold singular values
    s_thresh = s.copy()
    s_thresh[r_tilde:] = 0
    
    # Reconstruct filtered X
    X_train_filtered = U @ np.diag(s_thresh) @ Vt
    
    print(f"Filtered X shape: {X_train_filtered.shape}")
    
    # Fit OLS (no intercept)
    model = OLS(y_train, X_train_filtered).fit()
    beta = model.params
    
    print(f"Beta coefficients: {len(beta)} weights")
    print(f"Max |beta|: {np.max(np.abs(beta)):.2f}")
    print(f"R-squared: {model.rsquared:.4f}")
    
    X_remaining = historical_trajectories[:, T0:]            # (38, T1)
    X_remaining_for_pred = X_remaining.T                     # (T1, 38), here T1=150

    # Truncated SVD components
    U_r = U[:, :r_tilde]          # (150, r_tilde)
    s_r = s[:r_tilde]             # (r_tilde,)
    Vt_r = Vt[:r_tilde, :]        # (r_tilde, 38)

    # Project X_remaining_for_pred into the same low-rank space
    X_proj = X_remaining_for_pred @ Vt_r.T      # (T1, r_tilde)

    # Scale by singular values
    X_proj_scaled = X_proj * s_r                # Element-wise multiply (T1, r_tilde)

    # Truncate beta
    beta_r = beta[:r_tilde]                     # (r_tilde,)

    # Final prediction
    predictions = X_proj_scaled @ beta_r       
    
    print(f"Predictions shape: {predictions.shape}")
    
    # Calculate MSE
    mse = np.mean((predictions - target_remaining)**2)
    
    print(f"\nRESULTS:")
    print(f"MSE: {mse:.2f}")
    
    # Analyze prediction patterns
    errors = predictions - target_remaining
    
    # Phase analysis
    n_pred = len(predictions)
    beginning_errors = errors[:n_pred//4]
    end_errors = errors[-n_pred//4:]
    
    avg_beg_err = np.mean(beginning_errors)
    avg_end_err = np.mean(end_errors)
    
    print(f"\nTrajectory Analysis:")
    print(f"Beginning phase error: {avg_beg_err:.2f}")
    print(f"End phase error: {avg_end_err:.2f}")
    print(f"Under-estimates beginning: {avg_beg_err < 0}")
    print(f"Under-estimates end: {avg_end_err < 0}")
    
    # Volatility comparison
    actual_vol = np.std(np.diff(target_remaining))
    pred_vol = np.std(np.diff(predictions))
    
    print(f"Actual volatility: {actual_vol:.2f}")
    print(f"Predicted volatility: {pred_vol:.2f}")
    print(f"More fluctuating: {pred_vol > actual_vol}")
    
    return {
        'mse': mse,
        'max_beta': np.max(np.abs(beta)),
        'predictions': predictions,
        'actual': target_remaining,
        'under_beginning': avg_beg_err < 0,
        'under_end': avg_end_err < 0,
        'more_fluctuating': pred_vol > actual_vol
    }


def main():
    """Main execution with simple flow"""
    
    # Step 1: Clean data
    data = clean_cricket_data()
    if data is None:
        return
    
    target_trajectory, historical_trajectories = data
    
    # Step 2: Synthetic control
    results = synthetic_control_prediction(target_trajectory, historical_trajectories)
    
    # Step 3: Final answers
    print(f"\n{'='*50}")
    print("FINAL ANSWERS")
    print("="*50)
    
    print(f"Greatest β value: {results['max_beta']:.2f}")
    print(f"MSE: {results['mse']:.2f}")
    
    print(f"\nTrajectory statements (check which are true):")
    print(f"- Under-estimates at beginning: {results['under_beginning']}")
    print(f"- Under-estimates at end: {results['under_end']}")
    print(f"- More fluctuating: {results['more_fluctuating']}")

if __name__ == "__main__":
    main()

Filtered matches: 39
Score columns: 300
Target match: 1 record
Historical matches: 38
Cleaned target shape: (300,)
Cleaned historical shape: (38, 300)
Target score at T0=150: 128.0
Target final score: 239.0

SYNTHETIC CONTROL PREDICTION
Training period: 150 time points
Prediction period: 150 time points
Historical train shape: (38, 150)
Historical remaining shape: (38, 150)
OLS setup: X=(150, 38), y=(150,)
SVD: U=(150, 38), s=(38,), Vt=(38, 38)
Singular values: [5302.57189189  332.28040157  184.15567315   99.04203156   86.4675419
   71.967601     59.86290792   48.45071836   43.35875738   38.85723176]
Filtered X shape: (150, 38)
Beta coefficients: 38 weights
Max |beta|: 0.16
R-squared: 0.9994


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 38 is different from 150)